# ARC-Hydra: CPU build of Pennyroyal v3

Private infrastructure experiment. Builds the pinned upstream runtime and accepted FlashInfer kernels, then verifies a fresh offline installation. **GPU inference, throughput, agent quality and leaderboard performance remain untested.**

Runtime: John Pezzulli / SGLang contributors. INT4 compatibility and CUDA overlay adaptation: Daniel Franzen, Apache-2.0; original notices retained. No models or game data are needed by this builder.

In [ ]:
from pathlib import Path
import subprocess
import sys
BUILD_SOURCES = {'build_overlay.sh': '#!/usr/bin/env bash\n# CPU-only, online build. The resulting delta still needs real GPU validation.\nset -euo pipefail\nSOURCE_FILES="$1"\nROOT=/tmp/hydra-serving-v3\nOUT=/kaggle/working/pennyroyal-v3-overlay\nmkdir -p "$ROOT" "$OUT/wheels" "$OUT/compiler-wheels" "$OUT/licenses"\nexec > >(tee -a /kaggle/working/v3-build.log) 2>&1\ntrap \'echo "BUILD FAILED at line $LINENO; partial artifacts retained" >&2\' ERR\nBASE=/kaggle/input/datasets/dfranzen/pennyroyal-v253\nif [[ ! -d "$BASE" ]]; then BASE=/kaggle/input/pennyroyal-v253; fi\n[[ -d "$BASE/wheels" ]]\nNOTEBOOK_PY="$(command -v python)"\n"$NOTEBOOK_PY" -c \'import sys; assert sys.version_info[:2]==(3,12); print(sys.version)\'\nexport UV_HTTP_TIMEOUT=300 UV_HTTP_RETRIES=3 UV_PYTHON_DOWNLOADS=never UV_LINK_MODE=copy\nexport PIP_DEFAULT_TIMEOUT=300 PIP_RETRIES=3 PYTHONNOUSERSITE=1\nunset PYTHONPATH UV_SYSTEM_PYTHON\nexport MAX_JOBS=2 CMAKE_BUILD_PARALLEL_LEVEL=2 CARGO_BUILD_JOBS=2 FLASHINFER_NVCC_THREADS=1\nexport TORCH_CUDA_ARCH_LIST=\'12.0+PTX\'\ncp "$SOURCE_FILES/pins.json" "$OUT/pins.json"\ncp "$SOURCE_FILES/cuda_overlay.py" "$SOURCE_FILES/package_overlay.py" "$OUT/"\ncp "$SOURCE_FILES/LICENSE.Franzen" "$OUT/licenses/Franzen-Apache-2.0"\n\n# Use the existing offline bundle\'s native uv; no global Python replacement.\n"$NOTEBOOK_PY" - "$BASE" "$ROOT" <<\'PY\'\nfrom pathlib import Path\nfrom zipfile import ZipFile\nimport sys\nbase,root=map(Path,sys.argv[1:])\nwheel=next((base/\'wheels\').glob(\'uv-*.whl\'))\nwith ZipFile(wheel) as z:\n    name=next(n for n in z.namelist() if n.endswith(\'/uv\'))\n    p=root/\'uv\';p.write_bytes(z.read(name));p.chmod(0o755)\nPY\nUV="$ROOT/uv"\n\n# Frozen micromamba artifact; resolve the compiler packages and retain explicit\n# package URLs/versions. This compiler is private experiment infrastructure.\n"$NOTEBOOK_PY" - "$SOURCE_FILES/pins.json" "$ROOT" <<\'PY\'\nfrom hashlib import sha256\nimport json\nfrom pathlib import Path\nfrom urllib.request import urlopen\nimport sys\npins=json.loads(Path(sys.argv[1]).read_text())[\'micromamba\']\nwith urlopen(pins[\'url\'],timeout=120) as response:data=response.read()\nassert sha256(data).hexdigest()==pins[\'sha256\'],\'micromamba checksum mismatch\'\np=Path(sys.argv[2])/\'micromamba\';p.write_bytes(data);p.chmod(0o755)\nPY\nexport MAMBA_ROOT_PREFIX="$ROOT/mamba"\n"$ROOT/micromamba" create -y -p "$ROOT/gcc" --override-channels -c conda-forge \'gxx_linux-64=15\'\n"$ROOT/micromamba" list -p "$ROOT/gcc" --explicit > "$OUT/compiler-explicit.txt"\nexport CC="$ROOT/gcc/bin/x86_64-conda-linux-gnu-gcc"\nexport CXX="$ROOT/gcc/bin/x86_64-conda-linux-gnu-g++"\nexport CUDAHOSTCXX="$CXX"\n"$CXX" --version\n\n# Keep the CUDA 13.3 build toolkit separate from Torch\'s CUDA-13.0 wheel deps.\n"$UV" venv --python "$NOTEBOOK_PY" "$ROOT/compiler"\n"$NOTEBOOK_PY" - "$SOURCE_FILES/pins.json" "$OUT/compiler.lock" <<\'PY\'\nimport json\nfrom pathlib import Path\nimport sys\npins=json.loads(Path(sys.argv[1]).read_text())[\'cuda\']\nassert all(pins.values())\nPath(sys.argv[2]).write_text(\'\'.join(f\'{k}=={v}\\n\' for k,v in pins.items()))\nPY\n"$UV" pip install --python "$ROOT/compiler/bin/python" -r "$OUT/compiler.lock"\n"$ROOT/compiler/bin/python" "$SOURCE_FILES/cuda_overlay.py" "$ROOT/cuda"\nexport CUDA_HOME="$ROOT/cuda" CUDA_PATH="$ROOT/cuda" CUDACXX="$ROOT/cuda/bin/nvcc"\nexport PATH="$ROOT/gcc/bin:$ROOT/cuda/bin:$PATH"\nexport LD_LIBRARY_PATH="$ROOT/gcc/lib:$ROOT/cuda/lib64:${LD_LIBRARY_PATH:-}"\ncat > "$ROOT/probe.cu" <<\'CUDA\'\n#include <cuda_runtime.h>\n#include <cuda/std/array>\n__global__ void probe(float* x) { x[threadIdx.x] = 1.0f; }\nCUDA\n"$CUDACXX" -ccbin "$CXX" -arch=sm_120f -std=c++17 -cubin "$ROOT/probe.cu" -o "$ROOT/probe.cubin"\necho \'CPU CUDA SM120f compiler probe passed\'\n\n# Exact source. Carry only the same inspected INT4 scale-dtype fix used by the\n# original notebook; all other runtime changes are upstream release v3.\ngit clone --depth 1 --branch pennyroyal-v3.0.0 https://github.com/jpezzulli/sglang-rtxpro6000.git "$ROOT/source"\n[[ "$(git -C "$ROOT/source" rev-parse HEAD)" == 560edbdca18aedae13f71ae20b3984efa4d22633 ]]\n"$NOTEBOOK_PY" - "$ROOT/source" <<\'PY\'\nfrom pathlib import Path\nimport sys\np=Path(sys.argv[1])/\'python/sglang/srt/layers/quantization/gptq/schemes/gptq_moe.py\'\nb=p.read_bytes();assert b.count(b\'dtype=torch.half\')==2,\'Unexpected INT4 scale allocations\'\np.write_bytes(b.replace(b\'dtype=torch.half\',b\'dtype=params_dtype\'))\nPY\ngit -C "$ROOT/source" diff > "$OUT/int4-compatibility.patch"\ncp "$ROOT/source/LICENSE" "$OUT/licenses/sglang-LICENSE"\n\nexport CARGO_HOME="$ROOT/cargo" RUSTUP_HOME="$ROOT/rustup"\nexport PATH="$CARGO_HOME/bin:$PATH"\nif ! command -v cargo >/dev/null; then\n  curl --fail --location --retry 2 https://sh.rustup.rs -o "$ROOT/rustup-init.sh"\n  sh "$ROOT/rustup-init.sh" -y --profile minimal --no-modify-path\nfi\ncargo --version\nexport RUSTFLAGS=\'-C target-cpu=x86-64\'\nexport CARGO_TARGET_X86_64_UNKNOWN_LINUX_GNU_LINKER="$CC"\nexport SGLANG_BUILD_RUST_EXTS=all SETUPTOOLS_SCM_PRETEND_VERSION_FOR_SGLANG=0.5.19+g560edbdca18a\n"$UV" build --wheel "$ROOT/source/python" --python "$NOTEBOOK_PY" -o "$OUT/wheels"\n\n# Install the known working dependency base, then resolve only what the new\n# runtime actually needs. Freeze the delta, never ship unpinned online installs.\n"$UV" venv --python "$NOTEBOOK_PY" "$ROOT/runtime"\nRUNTIME="$ROOT/runtime/bin/python"\n"$UV" pip install --python "$RUNTIME" --no-index --find-links "$BASE/wheels" -r "$BASE/requirements.lock"\n"$RUNTIME" "$SOURCE_FILES/package_overlay.py" freeze "$ROOT/base-packages.json"\n"$UV" pip install --python "$RUNTIME" --prerelease=allow --index-strategy unsafe-best-match \\\n  --extra-index-url https://docs.sglang.ai/whl/cu130/ \\\n  --constraint "$ROOT/source/docker/pennyroyal/constraints.txt" "$OUT"/wheels/sglang-*.whl\n"$UV" pip install --python "$RUNTIME" --no-deps --index-url https://flashinfer.ai/whl/cu130 \\\n  \'flashinfer-jit-cache==0.7.0.post1+cu130\' \'flashinfer-jit-cache-sm120f==0.7.0.post1+cu130\'\n"$UV" pip install --python "$RUNTIME" --no-deps --index-url https://flashinfer.ai/whl \\\n  \'flashinfer-cubin==0.7.0.post1\'\n"$RUNTIME" -m pip check\n"$RUNTIME" "$ROOT/source/scripts/pennyroyal/flashinfer/install.py"\n"$RUNTIME" "$ROOT/source/scripts/pennyroyal/flashinfer/install.py" --check\n"$RUNTIME" "$SOURCE_FILES/package_overlay.py" package "$OUT" "$ROOT/base-packages.json"\n"$RUNTIME" -m pip download --only-binary=:all: --no-deps -r "$OUT/download.lock" \\\n  --extra-index-url https://docs.sglang.ai/whl/cu130/ \\\n  --extra-index-url https://flashinfer.ai/whl/cu130 \\\n  --extra-index-url https://flashinfer.ai/whl -d "$OUT/wheels"\n"$RUNTIME" -m pip download --only-binary=:all: --no-deps -r "$OUT/compiler.lock" -d "$OUT/compiler-wheels"\n\n# A fresh offline reinstall is mandatory. A completed build without these\n# checks is not an installable runtime, and none of this proves GPU inference.\n"$UV" venv --python "$NOTEBOOK_PY" "$ROOT/verify"\nVERIFY="$ROOT/verify/bin/python"\n"$UV" pip install --python "$VERIFY" --no-index --find-links "$BASE/wheels" -r "$BASE/requirements.lock"\n"$UV" pip install --python "$VERIFY" --no-index --find-links "$BASE/wheels" --find-links "$OUT/wheels" \\\n  -r "$OUT/requirements-overlay.lock"\n"$VERIFY" -m pip check\n"$VERIFY" "$ROOT/source/scripts/pennyroyal/flashinfer/install.py" --check\n"$VERIFY" -c \'import sglang,torch,flashinfer; print(sglang.__version__,torch.__version__,flashinfer.__version__)\'\ntar -C "$ROOT" -czf "$OUT/gcc-toolchain.tar.gz" gcc\n"$NOTEBOOK_PY" - "$ROOT" "$OUT" <<\'PY\'\nimport json,shutil,sys\nfrom pathlib import Path\nroot,out=map(Path,sys.argv[1:])\nfor p in (root/\'mamba/pkgs\').glob(\'*/info/licenses\'):\n    shutil.copytree(p,out/\'licenses/toolchain\'/p.parent.parent.name,dirs_exist_ok=True)\nprovenance=dict(source_commit=\'560edbdca18aedae13f71ae20b3984efa4d22633\',\n                base_dataset=\'dfranzen/pennyroyal-v253\',\n                validation=\'CPU build, compiled SM120f module and fresh offline installation; GPU inference unverified\',\n                int4_patch=\'Daniel Franzen INT4 scale-dtype compatibility fix, Apache-2.0\')\n(out/\'provenance.json\').write_text(json.dumps(provenance,indent=2)+\'\\n\')\nPY\n"$RUNTIME" "$SOURCE_FILES/package_overlay.py" manifest "$OUT"\necho \'CPU_BUILD_AND_OFFLINE_INSTALL_VERIFIED; GPU_INFERENCE_NOT_TESTED\' > "$OUT/BUILD_COMPLETE"\necho \'Pennyroyal v3 overlay build complete; GPU validation still required\'\n', 'cuda_overlay.py': '"""Assemble compiler wheels into a CUDA toolkit prefix.\n\nAdapted from Daniel Franzen\'s Apache-2.0 make_cuda_overlay.py, distributed\nwith arc-agi-3-solution. License: baselines/duck_m2/LICENSE.\n"""\nimport importlib.metadata as md\nfrom pathlib import Path\nimport subprocess\nimport sys\n\nroot=Path(sys.argv[1]).absolute()\nroot.mkdir()\nfor dist in sorted(md.distributions(),key=lambda d:d.metadata[\'Name\']):\n    if not dist.metadata[\'Name\'].lower().replace(\'_\',\'-\').startswith(\'nvidia-\'):\n        continue\n    for entry in dist.files or ():\n        parts=entry.parts\n        if len(parts)<4 or parts[0]!=\'nvidia\' or parts[2] not in (\'bin\',\'include\',\'lib\',\'lib64\',\'nvvm\'):\n            continue\n        source=Path(dist.locate_file(entry)).absolute()\n        if not source.is_file():\n            continue\n        dest=root/(\'lib64\' if parts[2]==\'lib\' else parts[2])/Path(*parts[3:])\n        dest.parent.mkdir(parents=True,exist_ok=True)\n        if dest.exists() or dest.is_symlink():\n            if dest.read_bytes()!=source.read_bytes():\n                raise RuntimeError(f\'Conflicting CUDA files: {dest}\')\n        else:\n            dest.symlink_to(source)\n(root/\'lib\').symlink_to(\'lib64\')\nfor name in (\'bin/nvcc\',\'bin/ptxas\',\'include/cuda.h\',\'include/cuda_runtime.h\',\n             \'include/crt/host_config.h\',\'nvvm/libdevice/libdevice.10.bc\'):\n    assert (root/name).is_file(),name\nassert (root/\'include/cccl/cuda/std\').is_dir()\nversion=subprocess.check_output([str(root/\'bin/nvcc\'),\'--version\'],text=True)\nassert \'release 13.3,\' in version,version\nprint(version)\n', 'package_overlay.py': '"""Freeze the built runtime delta and repack modified FlashInfer payloads."""\nfrom email.parser import Parser\nfrom hashlib import sha256\nimport importlib.metadata as md\nimport json\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\n\n\ndef packages():\n    return {re.sub(r\'[-_.]+\',\'-\',d.metadata[\'Name\']).lower():d.version for d in md.distributions()}\n\n\ndef repack(name, output, compiled=False):\n    dist=md.distribution(name)\n    stage=Path(\'/tmp/hydra-serving-v3/repack\')/name\n    stage.mkdir(parents=True)\n    tops={entry.parts[0] for entry in dist.files or () if entry.parts and entry.parts[0]!=\'..\'}\n    for top in tops:\n        source=Path(dist.locate_file(top))\n        if source.is_dir():\n            shutil.copytree(source,stage/top,ignore=shutil.ignore_patterns(\'__pycache__\'))\n        elif source.is_file():\n            shutil.copy2(source,stage/top)\n    wheel=next(stage.glob(\'*.dist-info/WHEEL\'))\n    if compiled:\n        data=Parser().parsestr(wheel.read_text())\n        del data[\'Tag\']\n        data[\'Tag\']=\'cp312-cp312-linux_x86_64\'\n        data.replace_header(\'Root-Is-Purelib\',\'false\')\n        wheel.write_text(str(data))\n    subprocess.run([sys.executable,\'-m\',\'wheel\',\'pack\',str(stage),\'--dest-dir\',str(output)],check=True)\n\n\nif __name__==\'__main__\':\n    command=sys.argv[1]\n    if command==\'freeze\':\n        Path(sys.argv[2]).write_text(json.dumps(packages(),indent=2)+\'\\n\')\n    elif command==\'package\':\n        root=Path(sys.argv[2]);base=json.loads(Path(sys.argv[3]).read_text())\n        after=packages();delta={k:v for k,v in after.items() if base.get(k)!=v}\n        assert delta[\'sglang\']==\'0.5.19+g560edbdca18a\'\n        assert delta[\'flashinfer-python\']==\'0.7.0.post1\'\n        (root/\'delta.json\').write_text(json.dumps(delta,indent=2)+\'\\n\')\n        (root/\'requirements-overlay.lock\').write_text(\'\'.join(f\'{k}=={v}\\n\' for k,v in sorted(delta.items())))\n        custom={\'sglang\',\'flashinfer-python\',\'flashinfer-cubin\'}\n        (root/\'download.lock\').write_text(\'\'.join(f\'{k}=={v}\\n\' for k,v in sorted(delta.items()) if k not in custom))\n        repack(\'flashinfer-python\',root/\'wheels\',compiled=True)\n        repack(\'flashinfer-cubin\',root/\'wheels\')\n    elif command==\'manifest\':\n        root=Path(sys.argv[2]);entries={}\n        for path in sorted(root.rglob(\'*\')):\n            if path.is_file() and path.name!=\'manifest.json\':\n                digest=sha256()\n                with path.open(\'rb\') as stream:\n                    for block in iter(lambda:stream.read(8*1024*1024),b\'\'):\n                        digest.update(block)\n                entries[path.relative_to(root).as_posix()]=dict(sha256=digest.hexdigest(),bytes=path.stat().st_size)\n        (root/\'manifest.json\').write_text(json.dumps(dict(files=entries),indent=2)+\'\\n\')\n', 'pins.json': '{\n  "source_commit": "560edbdca18aedae13f71ae20b3984efa4d22633",\n  "cuda": {\n    "nvidia-cuda-nvcc": "13.3.73",\n    "nvidia-cuda-crt": "13.3.73",\n    "nvidia-nvvm": "13.3.73",\n    "nvidia-cuda-runtime": "13.3.29",\n    "nvidia-cuda-cccl": "13.3.4.3.1"\n  },\n  "micromamba": {\n    "version": "2.9.0-0",\n    "url": "https://github.com/mamba-org/micromamba-releases/releases/download/2.9.0-0/micromamba-linux-64",\n    "sha256": "366cd9cd8be14df1ab8ed50352a82111082a36686b2d389fdb79a92c3fafb3e3"\n  }\n}', 'LICENSE.Franzen': '\n                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n'}
stage=Path('/kaggle/working/build-src')
stage.mkdir(parents=True,exist_ok=True)
for name,content in BUILD_SOURCES.items():
    (stage/name).write_text(content,encoding='utf-8')
subprocess.run(['bash',str(stage/'build_overlay.sh'),str(stage)],check=True)
